### 1. Set up Groq API Client
We will use the Groq client with your provided API key.

In [1]:
from groq import Groq

# Using your provided Groq API key
GROQ_API_KEY = "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"

try:
    # Using groq_client to prevent namespace collisions with Google's client
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client successfully initialized!")
except Exception as e:
    print(f"Error initializing Groq client: {e}")

Gemini client successfully initialized!


### 1. Set up Gemini API Client
To use the nutrition agent, make sure you have added your `GOOGLE_API_KEY` to Colab's secrets (the key icon on the left panel).

In [39]:
from google import genai
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("Gemini client successfully initialized!")
except Exception as e:
    print(f"Error initializing Gemini client: {e}\nMake sure you have set 'GOOGLE_API_KEY' in your Colab Secrets.")

Gemini client successfully initialized!


### Alternative: Enter API Key Manually
If you do not want to use Colab Secrets, run the block below to prompt you for your key manually.

In [40]:
import getpass
from google import genai

# Using the updated Google API key provided
GOOGLE_API_KEY = "AQ.Ab8RN6Lg-7X0Q1fQb22P4U5spldUjo3EDB4atAzHDaWxBXU3RA"

try:
    client = genai.Client(api_key=GOOGLE_API_KEY)
    print("Gemini client successfully initialized with your updated key!")
except Exception as e:
    print(f"Initialization failed: {e}")

Gemini client successfully initialized with your updated key!


### 2. Define the Shopping Catalog & Nutritional Data
Here is a list of common products with their nutritional values (calories, protein, carbs, fats, and category).

In [41]:
import pandas as pd

# Mock catalog of grocery products with nutritional information per 100g / serving
products_catalog = [
    {"product_id": 101, "name": "Greek Yogurt (Plain)", "category": "Dairy", "calories": 59, "protein_g": 10, "carbs_g": 3.6, "fat_g": 0.4},
    {"product_id": 102, "name": "Whole Milk", "category": "Dairy", "calories": 61, "protein_g": 3.2, "carbs_g": 4.8, "fat_g": 3.3},
    {"product_id": 103, "name": "Oats (Rolled)", "category": "Grains", "calories": 389, "protein_g": 16.9, "carbs_g": 66.3, "fat_g": 6.9},
    {"product_id": 104, "name": "White Bread", "category": "Bakery", "calories": 265, "protein_g": 9, "carbs_g": 49, "fat_g": 3.2},
    {"product_id": 105, "name": "Whole Wheat Bread", "category": "Bakery", "calories": 247, "protein_g": 13, "carbs_g": 41, "fat_g": 3.4},
    {"product_id": 106, "name": "Chicken Breast (Raw)", "category": "Meat/Poultry", "calories": 120, "protein_g": 22.5, "carbs_g": 0, "fat_g": 2.6},
    {"product_id": 107, "name": "Salmon Fillet", "category": "Seafood", "calories": 208, "protein_g": 20, "carbs_g": 0, "fat_g": 13},
    {"product_id": 108, "name": "Apple", "category": "Fruits", "calories": 52, "protein_g": 0.3, "carbs_g": 14, "fat_g": 0.2},
    {"product_id": 109, "name": "Banana", "category": "Fruits", "calories": 89, "protein_g": 1.1, "carbs_g": 23, "fat_g": 0.3},
    {"product_id": 110, "name": "Broccoli", "category": "Vegetables", "calories": 34, "protein_g": 2.8, "carbs_g": 7, "fat_g": 0.4},
    {"product_id": 111, "name": "Almonds", "category": "Snacks/Nuts", "calories": 579, "protein_g": 21, "carbs_g": 22, "fat_g": 49},
    {"product_id": 112, "name": "Potato Chips", "category": "Snacks/Nuts", "calories": 536, "protein_g": 7, "carbs_g": 53, "fat_g": 35}
]

df_catalog = pd.DataFrame(products_catalog)
display(df_catalog)

,product_id,name,category,calories,protein_g,carbs_g,fat_g
0,101,Greek Yogurt (Plain),Dairy,59,10.0,3.6,0.4
1,102,Whole Milk,Dairy,61,3.2,4.8,3.3
2,103,Oats (Rolled),Grains,389,16.9,66.3,6.9
3,104,White Bread,Bakery,265,9.0,49.0,3.2
4,105,Whole Wheat Bread,Bakery,247,13.0,41.0,3.4
5,106,Chicken Breast (Raw),Meat/Poultry,120,22.5,0.0,2.6
6,107,Salmon Fillet,Seafood,208,20.0,0.0,13.0
7,108,Apple,Fruits,52,0.3,14.0,0.2
8,109,Banana,Fruits,89,1.1,23.0,0.3
9,110,Broccoli,Vegetables,34,2.8,7.0,0.4


### 3. Build the Groq-Powered Nutrition Agent
We will now write the agent function using the `groq` SDK.

In [47]:
def run_nutrition_agent(user_query, cart_items=None):
    # Format the catalog as text to inject into the model prompt
    catalog_str = df_catalog.to_string(index=False)

    system_instruction = (
        "You are an expert Nutrition Agent integrated into a shopping application. "
        "Your job is to assist users with their shopping list, analyze nutritional choices, "
        "provide total calories and macros when relevant, and suggest healthier alternatives from the catalog if they choose unhealthy options."
    )

    prompt = f"""Here is the available shopping catalog with nutritional values (per 100g):
{catalog_str}

"""
    if cart_items:
        prompt += f"The user currently has these items in their cart: {cart_items}\n"

    prompt += f"User Query: {user_query}\n\nProvide a helpful, concise, and structured nutrition-focused response."

    try:
        # Updated model to llama-3.3-70b-versatile
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": system_instruction,
                },
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama-3.3-70b-versatile",
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Could not connect to Groq API. Error: {e}"

### 3. Build the Nutrition Agent
We will now write a function that takes a list of products (a simulated cart) or a general user nutrition question, provides context using our catalog, and calls Gemini to act as a personal Nutrition Advisor.

In [48]:
# Overwriting this cell to match the Groq-powered implementation using groq_client
def run_nutrition_agent(user_query, cart_items=None):
    catalog_str = df_catalog.to_string(index=False)

    system_instruction = (
        "You are an expert Nutrition Agent integrated into a shopping application. "
        "Your job is to assist users with their shopping list, analyze nutritional choices, "
        "provide total calories and macros when relevant, and suggest healthier alternatives from the catalog if they choose unhealthy options."
    )

    prompt = f"""Here is the available shopping catalog with nutritional values (per 100g):
{catalog_str}

"""
    if cart_items:
        prompt += f"The user currently has these items in their cart: {cart_items}\n"

    prompt += f"User Query: {user_query}\n\nProvide a helpful, concise, and structured nutrition-focused response."

    try:
        # Updated model to llama-3.3-70b-versatile
        chat_completion = groq_client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": system_instruction,
                },
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="llama-3.3-70b-versatile",
        )
        return chat_completion.choices[0].message.content
    except Exception as e:
        return f"Could not connect to Groq API. Error: {e}"

Nutrition Agent function has been updated with the Gemini configuration above!


### 4. Let's Test the Nutrition Agent
Let's run a test where a user asks for advice on a lunch order including potato chips and white bread.

In [50]:
test_cart = ["White Bread", "Potato Chips", "Greek Yogurt (Plain)"]
test_query = "Is this a healthy lunch selection? If not, what should I swap to keep it high protein and lower in calories?"

# Call the newly updated Groq-powered agent function
feedback = run_nutrition_agent(test_query, cart_items=test_cart)
print(feedback)

### **Nutrition Assessment**

Your current lunch selection is **not ideal** if your goals are high protein and lower calories. 

* **The Good:** **Greek Yogurt** is an excellent choice—low in calories and rich in protein.
* **The Drawbacks:** **Potato Chips** are calorie-dense and high in fat with minimal protein, while **White Bread** consists mostly of refined carbohydrates.

---

### **Current Cart Breakdown (per 100g)**

| Item | Calories | Protein | Carbs | Fat |
| :--- | :--- | :--- | :--- | :--- |
| Greek Yogurt (Plain) | 59 kcal | 10.0g | 3.6g | 0.4g |
| White Bread | 265 kcal | 9.0g | 49.0g | 3.2g |
| Potato Chips | 536 kcal | 7.0g | 53.0g | 35.0g |
| **Total** | **860 kcal** | **26.0g** | **105.6g** | **38.6g** |

---

### **Recommended Swaps from the Catalog**

To significantly cut calories and boost protein, consider these replacements:

1. **Swap Potato Chips ➔ Chicken Breast (Raw)**
   * *Why:* Drops **416 calories** while adding **+15.5g protein** per 100g, turning your 

In [ ]:
def run_nutrition_agent(user_query, cart_items=None):
    catalog_str = df_catalog.to_string(index=False)

    system_instruction = (
        "You are an expert Nutrition Agent integrated into a shopping application. "
        "Your job is to assist users with their shopping list, analyze nutritional choices, "
        "provide total calories and macros when relevant, and suggest healthier alternatives from the catalog if they choose unhealthy options."
    )

    prompt = f"""Here is the available shopping catalog with nutritional values (per 100g):
{catalog_str}

"""
    if cart_items:
        prompt += f"The user currently has these items in their cart: {cart_items}\n"

    prompt += f"User Query: {user_query}\n\nProvide a helpful, concise, and structured nutrition-focused response."

    try:
        # Using Gemini 2.5 Flash through the successfully initialized GenAI client
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config={
                'system_instruction': system_instruction
            }
        )
        return response.text
    except Exception as e:
        return f"Could not connect to Gemini API. Error: {e}"

# Run the test query with the active Gemini configuration
feedback = run_nutrition_agent(test_query, cart_items=test_cart)
print(feedback)